# Virtual Peripheral Binding Smoke Tests

Safe smoke-test cells for virtual peripherals. Run the setup cell first, then run one peripheral cell at a time.


In [9]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from contextlib import suppress
from pathlib import Path
import os
import sys
import time
from typing import Any


os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)


def find_workspace_root() -> Path:
    """
    Return the workspace root that contains the evomachine repository.

    Parameters
    ----------
    None

    Returns
    -------
    Path
        Workspace root used to add local sibling repositories to sys.path.
    """
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "evomachine" / "evomachine").is_dir():
            return candidate
    return current


def add_import_path(path: Path) -> None:
    """
    Add an existing package root to sys.path if it is not already present.

    Parameters
    ----------
    path
        Candidate package root to add for notebook imports.

    Returns
    -------
    None
    """
    if path.exists():
        path_text = str(path.resolve())
        if path_text not in sys.path:
            sys.path.insert(0, path_text)


def require_object(name: str) -> Any:
    """
    Return a global object by name or raise a clear notebook-order error.

    Parameters
    ----------
    name
        Global variable name expected to contain an initialised controller or peripheral.

    Returns
    -------
    Any
        The object stored under name in the notebook global namespace.
    """
    value = globals().get(name)
    if value is None:
        raise RuntimeError(f"Run the {name} setup cell before this cell.")
    return value


def remember_peripheral(name: str, peripheral: Any) -> Any:
    """
    Store a peripheral globally and add it to the cleanup list.

    Parameters
    ----------
    name
        Global variable name that should point to the peripheral.
    peripheral
        Peripheral instance created by a smoke-test cell.

    Returns
    -------
    Any
        The same peripheral instance, for inline assignment.
    """
    globals()[name] = peripheral
    if peripheral not in created_peripherals:
        created_peripherals.append(peripheral)
    return peripheral


def remember_controller(name: str, controller: Any) -> Any:
    """
    Store a controller globally and add it to the cleanup list.

    Parameters
    ----------
    name
        Global variable name that should point to the controller.
    controller
        Peripheral controller instance created by a smoke-test cell.

    Returns
    -------
    Any
        The same controller instance, for inline assignment.
    """
    globals()[name] = controller
    if controller not in created_controllers:
        created_controllers.append(controller)
    return controller


WORKSPACE_ROOT = find_workspace_root()
MAIN_REPO_ROOT = WORKSPACE_ROOT / "evomachine"
add_import_path(MAIN_REPO_ROOT)


from evomachine.bindings.virtual.autofocus import VirtualAutofocus
from evomachine.bindings.virtual.camera import VirtualCamera
from evomachine.bindings.virtual.filterwheel import VirtualFilterWheel
from evomachine.bindings.virtual.leds import VirtualLedSource
from evomachine.bindings.virtual.peripheralcontroller import VirtualPeripheralController
from evomachine.bindings.virtual.photodiode import VirtualPhotodiode
from evomachine.bindings.virtual.stage import VirtualStage

from evomachine.coordinates import Coordinate
from evomachine.peripherals.photodiode import PhotodiodeReadingRange
from evomachine.types import FilterWheelType, LEDType


FOV_STEP_SIZE = 100
STAGE_SMOKE_COORDINATE = Coordinate(x=10, y=100, z=-50)
RUN_STAGE_MOVE = False

VIRTUAL_TEST_LED = LEDType.LED_OVERHEAD_TIGER

LED_TEST_BRIGHTNESS = 1
LED_TEST_DURATION_MS = 100

RUN_FILTER_CHANGE = False
TARGET_FILTER = FilterWheelType.FILTER_527nm

RUN_AUTOFOCUS_CALIBRATION = False
LOCK_AFTER_AUTOFOCUS_CALIBRATION = False

PHOTODIODE_CHANNEL = 8
PHOTODIODE_READING_RANGE = PhotodiodeReadingRange(0.0, 1.0)

virtual_controller = None
virtual_stage = None
virtual_filter_wheel = None
virtual_led_source = None
virtual_autofocus = None
virtual_photodiode = None
created_controllers = []
created_peripherals = []

{
    "workspace_root": WORKSPACE_ROOT,
    "main_repo_root": MAIN_REPO_ROOT,
}


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


{'workspace_root': PosixPath('/home/hslab/workspace_python/evomachine_refactor'),
 'main_repo_root': PosixPath('/home/hslab/workspace_python/evomachine_refactor/evomachine')}

# Virtual Peripherals


In [10]:
# VirtualPeripheralController

virtual_controller = remember_controller(
    "virtual_controller",
    VirtualPeripheralController(),
)
virtual_controller.initialise()

{
    "name": virtual_controller.name,
    "is_initialised": virtual_controller.is_initialised(),
    "is_alive": virtual_controller.is_alive()

}


{'name': 'Virtual Peripheral Controller',
 'is_initialised': True,
 'is_alive': True}

In [11]:
# VirtualStage

virtual_controller = require_object("virtual_controller")
virtual_stage = remember_peripheral(
    "virtual_stage",
    VirtualStage(peripheral_ctrl=virtual_controller, fov_step_size=FOV_STEP_SIZE),
)
virtual_stage.initialise()
initial_coordinate = virtual_stage.get_coordinates()
limits = virtual_stage.get_stage_limits()

if RUN_STAGE_MOVE:
    virtual_stage.move(target=STAGE_SMOKE_COORDINATE, block=True)

{
    "name": virtual_stage.name,
    "is_initialised": virtual_stage.is_initialised(),
    "is_alive": virtual_stage.is_alive(),
    "inital_coordinate": initial_coordinate,
    "current_coordinate": virtual_stage.get_coordinates(),
    "stage_limits": limits,
    "ran_stage_move": RUN_STAGE_MOVE

}



2026-06-30 13:37:00 - DEBUG - evomachine.peripherals.stage - Stage.initialise: initialising Virtual Stage with force=False.
2026-06-30 13:37:00 - DEBUG - evomachine.peripherals.stage - Stage.initialise: Virtual Stage initialised at (x=0.0, y=0.0, z=0.0, channel_id=0).
2026-06-30 13:37:00 - DEBUG - evomachine.peripherals.stage - Stage._update_current_coordinate: Virtual Stage coordinate is now (x=0.0, y=0.0, z=0.0, channel_id=0).
2026-06-30 13:37:00 - DEBUG - evomachine.peripherals.stage - Stage._update_current_coordinate: Virtual Stage coordinate is now (x=0.0, y=0.0, z=0.0, channel_id=0).


{'name': 'Virtual Stage',
 'is_initialised': True,
 'is_alive': True,
 'inital_coordinate': Coordinate(x=0.0, y=0.0, z=0.0, channel_id=0),
 'current_coordinate': Coordinate(x=0.0, y=0.0, z=0.0, channel_id=0),
 'stage_limits': (Coordinate(x=-10000000.0, y=-10000000.0, z=-10000000.0, channel_id=0),
  Coordinate(x=10000000.0, y=10000000.0, z=10000000.0, channel_id=0)),
 'ran_stage_move': False}

In [12]:
# VirtualFilterWheel

virtual_controller = require_object("virtual_controller")
available_filters = list(FilterWheelType)
virtual_filter_wheel = remember_peripheral(
    "virtual_filter_wheel",
    VirtualFilterWheel(
        peripheral_ctrl=virtual_controller,
        available_filters=available_filters,
    ),
)
virtual_filter_wheel.initialise()
initial_filter = virtual_filter_wheel.get_filter_wheel()

if RUN_FILTER_CHANGE:
    virtual_filter_wheel.set_filter_wheel(filter_type=TARGET_FILTER, force=True)

{
    "name": virtual_filter_wheel.name,
    "is_initialised": virtual_filter_wheel.is_initialised(),
    "is_alive": virtual_filter_wheel.is_alive(),
    "available_filters": virtual_filter_wheel.get_available_filters(),
    "initial_filter": initial_filter,
    "current_filter": virtual_filter_wheel.get_filter_wheel(),
    "ran_filter_change": RUN_FILTER_CHANGE,
}


2026-06-30 13:37:00 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.initialise: initialising Virtual Filter Wheel with force=False.
2026-06-30 13:37:00 - DEBUG - evomachine.peripherals.filterwheel - FilterWheel.initialise: Virtual Filter Wheel initialised at UNKNOWN.


{'name': 'Virtual Filter Wheel',
 'is_initialised': True,
 'is_alive': True,
 'available_filters': [<FilterWheelType.UNKNOWN: -1>,
  <FilterWheelType.FILTER: 0>,
  <FilterWheelType.FILTER_465nm: 1>,
  <FilterWheelType.FILTER_527nm: 2>,
  <FilterWheelType.FILTER_592nm: 3>,
  <FilterWheelType.NO_FILTER: 4>,
  <FilterWheelType.BLOCKING: 5>],
 'initial_filter': <FilterWheelType.UNKNOWN: -1>,
 'current_filter': <FilterWheelType.UNKNOWN: -1>,
 'ran_filter_change': False}

In [13]:
# VirtualLedSource

virtual_controller = require_object("virtual_controller")
virtual_led_source = remember_peripheral(
    "virtual_led_source",
    VirtualLedSource(
        peripheral_ctrl=virtual_controller,
        available_leds=[VIRTUAL_TEST_LED],
    ),
)
virtual_led_source.initialise()
try:
    virtual_led_source.set_led(
        led_type=VIRTUAL_TEST_LED,
        brightness=LED_TEST_BRIGHTNESS,
        duration=LED_TEST_DURATION_MS,
    )
    time.sleep(LED_TEST_DURATION_MS / 1000.0 + 0.05)
finally:
    virtual_led_source.disable_led()

{
    "name": virtual_led_source.name,
    "is_initialised": virtual_led_source.is_initialised(),
    "is_alive": virtual_led_source.is_alive(),
    "available_leds": virtual_led_source.get_available_leds(),
    "test_led": VIRTUAL_TEST_LED,
    "brightness": LED_TEST_BRIGHTNESS,
    "duration_ms": LED_TEST_DURATION_MS,
    "state_after_disable": virtual_led_source.get_led_state(VIRTUAL_TEST_LED),
}


2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.leds - LedSource.initialise: initialising Virtual LED Source with force=False.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting Virtual LED Source LED_OVERHEAD_TIGER to brightness=1.0 duration=100.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: disabling LED_OVERHEAD_TIGER on Virtual LED Source.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: disabling LED_OVERHEAD_TIGER on Virtual LED Source.


{'name': 'Virtual LED Source',
 'is_initialised': True,
 'is_alive': True,
 'available_leds': [<LEDType.LED_OVERHEAD_TIGER: 6>],
 'test_led': <LEDType.LED_OVERHEAD_TIGER: 6>,
 'brightness': 1,
 'duration_ms': 100,
 'state_after_disable': LedState(led_type=<LEDType.LED_OVERHEAD_TIGER: 6>, brightness=0.0, is_on=False, stop_time=None)}

In [14]:
# VirtualAutofocus

virtual_controller = require_object("virtual_controller")
virtual_autofocus = remember_peripheral(
    "virtual_autofocus",
    VirtualAutofocus(peripheral_ctrl=virtual_controller),
)
virtual_autofocus.initialise()
initial_status = virtual_autofocus.get_status()
calibration_success = None

if RUN_AUTOFOCUS_CALIBRATION:
    calibration_success = virtual_autofocus.initialise_autofocus(
        lock_after_initialise=LOCK_AFTER_AUTOFOCUS_CALIBRATION,
    )

{
    "name": virtual_autofocus.name,
    "is_initialised": virtual_autofocus.is_initialised(),
    "is_alive": virtual_autofocus.is_alive(),
    "initial_status": initial_status,
    "current_status": virtual_autofocus.get_status(),
    "is_locked": virtual_autofocus.is_locked(),
    "ran_autofocus_calibration": RUN_AUTOFOCUS_CALIBRATION,
    "calibration_success": calibration_success,
}


2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.autofocus - Autofocus.initialise: initialising Virtual Autofocus with force=False.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.autofocus - Autofocus.initialise: Virtual Autofocus initialised.


{'name': 'Virtual Autofocus',
 'is_initialised': True,
 'is_alive': True,
 'initial_status': <AutoFocusStatusType.IDLE: 'I'>,
 'current_status': <AutoFocusStatusType.IDLE: 'I'>,
 'is_locked': False,
 'ran_autofocus_calibration': False,
 'calibration_success': None}

In [15]:
# VirtualPhotodiode

virtual_controller = require_object("virtual_controller")
virtual_photodiode = remember_peripheral(
    "virtual_photodiode",
    VirtualPhotodiode(
        peripheral_ctrl=virtual_controller,
        channel=PHOTODIODE_CHANNEL,
        reading_range=PHOTODIODE_READING_RANGE,
    ),
)
virtual_photodiode.initialise()
reading = virtual_photodiode.read_photodiode()

{
    "name": virtual_photodiode.name,
    "is_initialised": virtual_photodiode.is_initialised(),
    "is_alive": virtual_photodiode.is_alive(),
    "channel": virtual_photodiode.channel,
    "reading_percent": reading,
}


2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.photodiode - Photodiode.initialise: initialising Virtual Photodiode with force=False.


2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.photodiode - Photodiode.read_photodiode: Virtual Photodiode raw reading=0.0.


{'name': 'Virtual Photodiode',
 'is_initialised': True,
 'is_alive': True,
 'channel': 8,
 'reading_percent': 0.0}

## Cleanup

In [16]:
# Stop and release any peripherals/controllers created above.

cleanup_errors = []

for peripheral in reversed(created_peripherals):
    with suppress(Exception):
        peripheral.stop()
    try:
        peripheral.finalise()
    except Exception as error:
        cleanup_errors.append((getattr(peripheral, "name", repr(peripheral)), repr(error)))

for controller in reversed(created_controllers):
    try:
        controller.shutdown(force=True)
    except Exception as error:
        cleanup_errors.append((getattr(controller, "name", repr(controller)), repr(error)))

{
    "cleaned_peripherals": [getattr(peripheral, "name", repr(peripheral)) for peripheral in created_peripherals],
    "cleaned_controllers": [getattr(controller, "name", repr(controller)) for controller in created_controllers],
    "cleanup_errors": cleanup_errors,
}


2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.photodiode - Photodiode.stop: stopping Virtual Photodiode.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.photodiode - Photodiode.finalise: finalising Virtual Photodiode with force=False.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.autofocus - Autofocus.stop: stopping Virtual Autofocus.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.autofocus - Autofocus.disable: disabling Virtual Autofocus.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.autofocus - Autofocus.finalise: finalising Virtual Autofocus with force=False.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.leds - LedSource.stop: disabling all LEDs for Virtual LED Source.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: disabling LED_OVERHEAD_TIGER on Virtual LED Source.
2026-06-30 13:37:01 - DEBUG - evomachine.peripherals.leds - LedSource.finalise: finalising Virtual LED Source with force=False.
2026-06-30 13:37:

{'cleaned_peripherals': ['Virtual Stage',
  'Virtual Filter Wheel',
  'Virtual LED Source',
  'Virtual Autofocus',
  'Virtual Photodiode'],
 'cleaned_controllers': ['Virtual Peripheral Controller'],
 'cleanup_errors': []}